# Ollama Chat Model — Local Inference Architecture

## Overview

This notebook demonstrates how to use a locally running LLM via Ollama.

Everything runs on:

No external API.
No API key.
No rate limits.

The model is served locally through the Ollama runtime.

---

## Core Components

### 1️⃣ Ollama Server

Ollama must be running in the background.

It exposes REST endpoints such as:

- /api/generate
- /api/chat
- /api/embeddings

The notebook sends requests to these endpoints.

---

## Chat Request Structure

A chat request includes:

- model
- messages
- temperature (optional)
- stream (optional)
- other sampling parameters

---

## Parameters Used

### model
Specifies the local model name.

Example:
- gemma3:1b
- llama3
- mistral

Must match exact name from:

`ollama list`

---

### messages

Structured conversation history:

- role: system
- role: user
- role: assistant

Used to simulate multi-turn conversation.

---

### temperature

Controls randomness:

- 0.0 → deterministic
- 0.3–0.7 → balanced
- >1.0 → creative

Lower temperature is recommended for:
- Q&A
- RAG
- Structured responses

---

### stream (optional)

If true:
- Tokens are returned progressively
- Useful for real-time UI streaming

If false:
- Full response returned at once

---

## Response Structure

The model returns:

- model name
- created timestamp
- message
- content
- done flag

The generated answer is found inside:

`response["message"]["content"]`

---

## Practical Architecture (Local Chat System)

User Input  
→ Format into structured messages  
→ Send to Ollama /api/chat  
→ Model generates response  
→ Extract message content  
→ Display to user  

This is the minimal chat pipeline.

---

## Why This Matters

Using Ollama locally allows:

- Offline development
- Full privacy
- Zero API cost
- Custom model selection
- Control over sampling parameters

It forms the foundation for:

- Local agents
- RAG systems
- Tool-calling systems
- Multi-agent workflows

---

## Key Takeaway

Ollama acts as a local model server.

Your notebook:
- Formats messages
- Sends structured requests
- Receives structured responses
- Extracts generated content

This is the base architecture for building local AI applications.

In [20]:
import os 
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")

## Langsmith tracking
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "default_project")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [18]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model = "gpt-3.5-turbo")
llm

ChatOpenAI(profile={'max_input_tokens': 16385, 'max_output_tokens': 4096, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': False, 'structured_output': False, 'image_url_inputs': False, 'pdf_inputs': False, 'pdf_tool_message': False, 'image_tool_message': False, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x16229e7d0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x16229d090>, root_client=<openai.OpenAI object at 0x161fd7e50>, root_async_client=<openai.AsyncOpenAI object at 0x16229cc90>, model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)

In [21]:
from langchain_groq import ChatGroq

In [23]:
llm = ChatGroq(model = "gemma2-9b-it")
llm


ChatGroq(profile={'max_input_tokens': 8192, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x1625f1250>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x1625f1d10>, model_name='gemma2-9b-it', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3")
result= llm.invoke("What is the capital of France?")

In [28]:
print(result)

content='The capital of France is Paris.' additional_kwargs={} response_metadata={'model': 'llama3', 'created_at': '2026-02-18T19:19:35.266517Z', 'done': True, 'done_reason': 'stop', 'total_duration': 51828220417, 'load_duration': 12519664583, 'prompt_eval_count': 17, 'prompt_eval_duration': 38427093459, 'eval_count': 8, 'eval_duration': 675671335, 'logprobs': None, 'model_name': 'llama3', 'model_provider': 'ollama'} id='lc_run--019c7230-bf69-7280-922c-875c45eff3a7-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 17, 'output_tokens': 8, 'total_tokens': 25}


In [36]:
#Chatprompt template

from langchain_core.prompts import ChatPromptTemplate


prompt = ChatPromptTemplate.from_messages(
    [("system", "You are a helpful assistant that translates English to Arabic or vice versa.")
     ,("human", "{input}")]
)

prompt

ChatPromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant that translates English to Arabic or vice versa.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [37]:
## chaining

chain = prompt | llm

response= chain.invoke({"input": "Vous expliquez le difference entre deux et dois en françaos?"})



In [38]:
print (response)

content='I\'m happy to help! However, I noticed you asked about the difference between "deux" and "dois" in French.\n\nLet me clarify that for you:\n\n* "Deux" is the French word for "two". It\'s an indefinite article used with masculine nouns.\n* "Dois" is not a French word. You might be thinking of the verb "avoir" (to have) in the second person singular, which is "dois".\n\nIn French, when you say "je dois", it means "I must" or "I have to". For example: "Je dois aller au bureau demain" means "I have to go to the office tomorrow".\n\nSo, to summarize:\n\n* "Deux" = two (indefinite article with masculine nouns)\n* "Dois" = I have to/it\'s necessary (second person singular of the verb "avoir")\n\nI hope that clears up any confusion!' additional_kwargs={} response_metadata={'model': 'llama3', 'created_at': '2026-02-18T19:39:14.386948Z', 'done': True, 'done_reason': 'stop', 'total_duration': 53383113542, 'load_duration': 233467167, 'prompt_eval_count': 44, 'prompt_eval_duration': 397564

In [39]:
## stroutput parser

from langchain_core.output_parsers import StrOutputParser


parser = StrOutputParser()

chain = prompt | llm | parser

response = chain.invoke({"input": "Can you write the lyrics of  Jann Al hawa by wael koufri in Arabic and explain me the meaning of it in English?"})
print(response)

I'd be happy to help you with that!

The song "Jann Al-Hawa" (جان الحوا) is a popular Arabic song by Wael Koufi. Here are the lyrics in Arabic:

جان الحوا يهونني
أنا في نومي، ونومك في عيني
ما زال النوم يتهادي في روحي
ليوم ينسينا ويلحظنا

وأنا في حلمي، وحلمك في راسي
ما زال الحلم يتهادي في قلبي
ليوم ينسينا ويلحظنا

ما الذي يمنعنا من النوم
فينام ليلة واحدة فقط؟
ما الذي يمنعنا من النوم
فيأبقى ليلة واحدة أخرى؟

 Jana al-Hawa yihuni
Ana fi nuomi, wa nuomak fi aini
Maa zal al-nu'ma yitahadi fi ruhi
Li yawma yansina wa ilhazna

Wa ana fi halmi, wahalkum fi ras'i
Maa zal al-halima yitahadi fi qalbi
Li yawma yansina wa ilhazna

Maa dhalik allama'ana min al-nu'ma
Faynam liwa'at ukhra 'idda?
Maa dhalik allama'ana min al-nu'ma
Fi yabqi liwa'at ukhra 'udda?

Now, let me explain the meaning of the song in English:

The song "Jann Al-Hawa" (which translates to "The Soul of the Wind") is a poetic and romantic expression about the longing for unity with one's loved one. The lyrics describe the deep slee